# 🎵 ACE-Step 1.5 no Colab — versão 2 (uma célula só)
1. **Ambiente de execução → Alterar tipo → A100 GPU** (ou T4 e troque o modelo para `1.7B` na célula 2).
2. Rode a **célula 1** e depois a **célula 2**. No final aparece o link para abrir no navegador.
3. **Baixe as músicas assim que gerar** e, ao terminar, *Desconectar e excluir ambiente de execução*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, re, shutil, subprocess, time, urllib.request

# ===== CONFIGURAÇÃO =====
LM_MODEL = "acestep-5Hz-lm-4B"   # A100: 4B | T4/L4: troque para "acestep-5Hz-lm-1.7B"
REPO = "/content/ACE-Step-1.5"
PORT = 7860
# ========================

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, text=True, cwd=cwd,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return r.returncode, r.stdout

def tail(path, n=40):
    try:
        return "".join(open(path, errors="ignore").readlines()[-n:])
    except Exception as e:
        return f"(sem log: {e})"

print("1/5 Preparando o uv...")
if not shutil.which("uv"):
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh")
assert shutil.which("uv"), "uv não foi instalado"

print("2/5 Baixando o ACE-Step...")
if not os.path.isdir(REPO):
    code, out = sh(f"git clone https://github.com/ACE-Step/ACE-Step-1.5.git {REPO}")
    assert code == 0, out
os.chdir(REPO)

print("3/5 Instalando dependências (alguns minutos)...")
code, out = sh("uv sync", cwd=REPO)
if code != 0:
    print(out[-3000:])
    raise SystemExit("❌ Falha no uv sync (veja acima)")

with open(os.path.join(REPO, ".env"), "w") as f:
    f.write(f"ACESTEP_CONFIG_PATH=acestep-v15-turbo\nACESTEP_LM_MODEL_PATH={LM_MODEL}\nPORT={PORT}\nLANGUAGE=en\n")

print("4/5 Ligando a IA (na 1ª vez baixa os modelos, pode levar 5-15 min)...")
sh("pkill -f acestep_v15_pipeline")
logf = open("/content/acestep.log", "w")
proc = subprocess.Popen(
    ["uv", "run", "python", "-c", "from acestep.acestep_v15_pipeline import main; main()"],
    cwd=REPO, stdout=logf, stderr=subprocess.STDOUT,
)

ready = False
t0 = time.time()
for i in range(180):                      # até ~15 min
    if proc.poll() is not None:
        break
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}", timeout=3)
        ready = True
        break
    except Exception:
        time.sleep(5)
        if i % 12 == 11:
            print(f"   ...{int(time.time()-t0)}s | último log: " + (tail("/content/acestep.log", 1).strip()[:120] or "-"))

if not ready:
    print("\n❌ A IA não subiu. Diagnóstico (me mande um print disto):")
    print("processo ainda vivo:", proc.poll() is None)
    print("--- log ---")
    print(tail("/content/acestep.log", 40))
    print("--- .venv/bin ---")
    print(sh("ls .venv/bin | head -30", cwd=REPO)[1])
    raise SystemExit

print("✅ ACE-Step no ar!")

print("5/5 Criando o link para o navegador...")
cf = "/content/cloudflared"
if not os.path.exists(cf):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cf)
    os.chmod(cf, 0o755)
sh("pkill -f cloudflared")
tl = open("/content/tunnel.log", "w")
subprocess.Popen([cf, "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
                 stdout=tl, stderr=subprocess.STDOUT)

url = None
for _ in range(40):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tail("/content/tunnel.log", 200))
    if m:
        url = m.group(0)
        break
    time.sleep(2)

print("\n" + "=" * 60)
print("🎵 ABRA ESTE LINK NO NAVEGADOR:" if url else "⚠️ Link não apareceu. Log do túnel:")
print(url or tail("/content/tunnel.log", 20))
print("=" * 60)
print("Deixe esta aba do Colab aberta enquanto estiver usando.")

### Se algo der errado
Rode a célula abaixo e me mande um print:

In [ ]:
!echo '--- acestep.log ---'; tail -n 40 /content/acestep.log
!echo '--- tunnel.log ---'; tail -n 10 /content/tunnel.log